In [4]:
import json

with open('../data/task4.json') as f:
    data = json.load(f)


In [5]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# 1. Load environment variables from the .env file
load_dotenv()

# 2. Safely retrieve the API Key
SILICONFLOW_API_KEY = os.getenv("SILICONFLOW_API_KEY")

# Add a safety check to ensure the API key is loaded properly
if not SILICONFLOW_API_KEY:
    raise ValueError("API Key not found. Please ensure your .env file is created correctly and contains the SILICONFLOW_API_KEY variable!")

# 3. Configure the LLM using the SiliconFlow API
llm = ChatOpenAI(
    model="Pro/moonshotai/Kimi-K2.5", 
    api_key=SILICONFLOW_API_KEY,
    base_url="https://api.siliconflow.cn/v1", 
    temperature=0.0
    # No max_tokens parameter here, as the model needs to generate full statements
)

In [6]:
from tqdm import tqdm
from langchain_core.messages import HumanMessage, SystemMessage

def generate_statements(draft_data):
    # We use tqdm to track the progress of the outer loop (processing drafts)
    for draft in tqdm(draft_data, desc="Processing Drafts"):
        statements = draft['Statements']
        
        # Iterate through each statement requirement for the current draft
        for statement in statements:
            country = statement['country']
            
            # Strictly restored the original system and user prompts
            system_prompt = f"""
            Assume you are the representative of {country} in UNSC, given the content of a UNSC draft resolution, generate a statement that you would make in the meeting.
            """
            
            user_prompt = f"""
            Draft resolution: {draft['Content']}
            Your statement:
            """
            
            try:
                # Construct LangChain messages and invoke the model
                messages = [
                    SystemMessage(content=system_prompt),
                    HumanMessage(content=user_prompt)
                ]
                response = llm.invoke(messages)
                
                # Save the generated text to the dictionary
                statement['generation'] = response.content.strip()
                
            except Exception as e:
                # Catch API errors to prevent the loop from crashing
                print(f"Error generating statement for {country}: {e}")
                statement['generation'] = "" # Leave empty if an error occurs
                
        # Update the statements list in the draft dictionary
        draft['Statements'] = statements
        
    return draft_data

# Execute the function to generate statements and update the 'data' variable
data = generate_statements(data)

Processing Drafts: 100%|██████████| 4/4 [36:08<00:00, 542.12s/it] 


In [7]:
statements = []
generations = []

for draft in data:
    for statement in draft['Statements']:
        statements.append(statement['statement'])
        generations.append(statement['generation'])

In [8]:
# average ROUGE
from rouge_score import rouge_scorer
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

rouge_scores = []
for i in range(len(statements)):
    scores = scorer.score(statements[i], generations[i])
    rouge_scores.append(scores['rougeL'].fmeasure)

print('Average ROUGE-L:', sum(rouge_scores) / len(rouge_scores))

Average ROUGE-L: 0.1873322984414462


In [9]:
# average Jaccard similarity
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import jaccard_score

jaccard_scores = []
vectorizer = CountVectorizer(binary=True)

for i in range(len(statements)):
    X = vectorizer.fit_transform([statements[i], generations[i]]).toarray()
    jaccard_scores.append(jaccard_score(X[0], X[1]))

print('Average Jaccard similarity:', sum(jaccard_scores) / len(jaccard_scores))

Average Jaccard similarity: 0.16836977377015458


In [10]:
# average Cosing Similarity (TF-IDF)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tf_cosine_scores = []
for i in range(len(statements)):
    tfidf = TfidfVectorizer().fit_transform([statements[i], generations[i]])
    tf_cosine_scores.append(cosine_similarity(tfidf[0], tfidf[1])[0][0])

print('Average Cosine Similarity (TF-IDF):', sum(tf_cosine_scores) / len(tf_cosine_scores))

Average Cosine Similarity (TF-IDF): 0.7695563175954102


In [11]:
import torch
from sentence_transformers import SentenceTransformer, util

# 自动检测可用硬件加速：Apple Silicon 使用 mps，NVIDIA 使用 cuda，否则用 cpu
if torch.backends.mps.is_available():
    device = 'mps'
elif torch.cuda.is_available():
    device = 'cuda'
else:
    device = 'cpu'

print(f"Using device: {device}")
model = SentenceTransformer('stsb-roberta-large', device=device)
cosine_scores = []

for i in range(len(statements)):
    embeddings = model.encode([statements[i], generations[i]])
    cosine_scores.append(util.pytorch_cos_sim(embeddings[0], embeddings[1]).item())

print('Average Cosine Similarity (BERT):', sum(cosine_scores) / len(cosine_scores))

Using device: mps


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Average Cosine Similarity (BERT): 0.6364001640502144
